In [1]:
company_list = [
        'Accenture', 'Adobe', 'Cisco', 'Citi', 'DHL Express', 'Exxon Mobil', 'FedEx', 
        'General Electric', 'HSBC', 'Huawei', 'IBM', 'Intel', 'J.P. Morgan', 
        'Microsoft', 'Oracle', 'Salesforce', 'SAP', 'Shell', 'UPS', 'Wells Fargo'
    ]
non_biz_labels = ["Thought Leadership", "ESG & Purpose", "Employer Branding", "Public Relations & Lifestyle", "Macro/Crisis Response"]
label_list = ["Health & Public Service", "Products", "Financial Services", 
              "Digital Media", "Digital Marketing", "Digital Experience", "Print and Publishing",
              #"Product", "Service",
              "CITICORP", "Global Consumer Banking", "Personal Banking and Wealth Management",
              #"eCommerce Solutions", "Express",
              "Upstream", "Downstream", "Chemical Products",
              "FedEx Services segment", "FedEx Express segment", "FedEx Ground segment",
              "Aviation", "Healthcare", "Renewable Energy", "Transportation",
              "CMB", "RBWM", "Global Banking and Markets", 
              "Enterprise Business", "Carrier Business", "ICT Infrastructure Business", "Cloud Computing",
              "Software", "Cognitive Solutions", "Global Business Services", "Global Technology Services",
              "PC Client Group", "Client Computing Group", "Internet of Things Group", "Data Center Group",
              #"Corporate & Investment Bank", "Asset & Wealth Management", 
              "Windows PC operating systems", "Office products and cloud services", "Gaming",
              #"Cloud and license", "Hardware",
              "Subscription and support", "Professional services and other",
              #"Cloud and software", "Services", 
              #"Upstream", "Downstream",
              #"U.S. Domestic Package", "Supply Chain & Freight", "International Package",
              #"Community Banking", "Wealth, Brokerage and Retirement", "Wholesale Banking"
              ]
label_mapping = {"USPB": "U.S. Personal Banking",
                 "eCommerce": "eCommerce Solutions",
                 "Chemical": "Chemical Products",
                 "HealthCare": "Healthcare",
                 "GB&M": "Global Banking and Markets",
                 "GBM": "Global Banking and Markets",
                 "Carrier Network Business": "Carrier Business",
                 "Carrier Network": "Carrier Business",
                 "ICT Infrastructure": "ICT Infrastructure Business",
                 "Client Computing Group (CCG)": "Client Computing Group",
                 "Internet of Things Group (IOTG)": "Internet of Things Group",
                 "Data Center Group (DCG) ": "Data Center Group",
                 "Internet of Things": "Internet of Things Group",
                 "Asset Management": "Asset & Wealth Management",
                 "Windows PC operating system": "Windows PC operating systems"
                 }

In [ ]:
import os
import google.generativeai as genai
import google.api_core.client_options
from google.api_core import retry

# If using VPN, set Network proxy.
proxy_url = 'http://127.0.0.1:7897'
os.environ['HTTP_PROXY'] = proxy_url
os.environ['HTTPS_PROXY'] = proxy_url
os.environ['GRPC_PROXY_EXPAND_HOSTS'] = '1'
os.environ['grpc_proxy'] = proxy_url

client_options = google.api_core.client_options.ClientOptions(
    api_endpoint="generativelanguage.googleapis.com"
)
genai.configure(
    api_key="API_KEY",
    transport='rest',
    client_options=client_options 
)

In [ ]:
INSTRUCTION_NORM = """
You are a financial expert. Classify the given tweet into one of the following categories:
Categories: ["Revenue Segments", "Thought Leadership", "ESG & Purpose", "Employer Branding", "PR & Lifestyle", "Macro/Crisis Response", 
             "Health & Public Service", "Products", "Financial Services", 
              "Digital Media", "Digital Marketing", "Digital Experience", "Print and Publishing",
              "CITICORP", "Global Consumer Banking", "Personal Banking and Wealth Management",
              "Upstream", "Downstream", "Chemical Products",
              "FedEx Services segment", "FedEx Express segment", "FedEx Ground segment",
              "Aviation", "Healthcare", "Renewable Energy", "Transportation",
              "CMB", "RBWM", "Global Banking and Markets", 
              "Enterprise Business", "Carrier Business", "ICT Infrastructure Business", "Cloud Computing",
              "Software", "Cognitive Solutions", "Global Business Services", "Global Technology Services",
              "PC Client Group", "Client Computing Group", "Internet of Things Group", "Data Center Group",
              "Windows PC operating systems", "Office products and cloud services", "Gaming",
              "Subscription and support", "Professional services and other",]
Return the result in JSON format: {"label": "category_name"}
"""

In [ ]:
import pandas as pd
import os
import json
import time

train_path = r"data/tweet_label_train.csv"
test_path = r"data/tweet_label_test.csv"
train_df, test_df = pd.read_csv(train_path), pd.read_csv(test_path)

In [ ]:
DIR_JSON = r'data/fin_data'

label_describtion = {
        "Thought Leadership": "General insights/trends NOT tied to a specific business unit.",
        "ESG & Purpose": "Sustainability, CSR, or diversity.",
        "Employer Branding": "Recruitment and culture.",
        "Public Relations & Lifestyle": "General greetings, sponsorships, or PR.",
        "Macro/Crisis Response": "Geopolitics, economy, or disaster updates."
    }

def get_label_describtion(company_list, label_list, label_describtion, file_dir):    
    for company_name in company_list:
        company_path = os.path.join(file_dir, company_name)
        if not os.path.exists(company_path):
            continue
        
        for filename in os.listdir(company_path):
            if not filename.lower().endswith('.json'):
                continue

            json_path = os.path.join(company_path, filename)
            with open(json_path, 'r', encoding='utf-8-sig') as file:
                fin_data = json.load(file)

            revenue_segments = fin_data["revenue_segments"]
            for s in revenue_segments:
                segment_name = s["segment"]
                if segment_name in label_list: 
                    if "segment_description" in s.keys():
                        label_describtion[segment_name] = s["segment_description"]
                    else:
                        label_describtion[segment_name] = s["description"]

    return label_describtion

# Obtain contexts
label_describtion = get_label_describtion(company_list, label_list, label_describtion, DIR_JSON)

In [ ]:
from sklearn.metrics import classification_report

# ==========================================
# Setting
# ==========================================
generation_config = {
    "temperature": 0.0,
    "response_mime_type": "application/json",
}

# model_name: 'gemini-2.0-flash' or 'gemini-2.5-flash'
model_norm = genai.GenerativeModel(
    model_name='gemini-2.5-flash', 
    system_instruction=INSTRUCTION_NORM,
    generation_config=generation_config
)


# ==========================================
# 1. Build Context
# ==========================================
def get_context_str(label_describtion):
    context_str = "Below are the specific definitions for each category in this company's context:\n"
    for label, desc in label_describtion.items():
        context_str += f"- {label}: {desc}\n"
    return context_str

context_info = get_context_str(label_describtion)

def get_few_shot_examples(df, n_per_label=10):
    examples = df.groupby('label').apply(lambda x: x.sample(n_per_label)).reset_index(drop=True)
    example_str = "Below are some examples of correctly classified tweets:\n"
    for _, row in examples.iterrows():
        example_str += f"Tweet: {row['tweet']}\nLabel: {row['label']}\n---\n"
    return example_str

few_shot_examples = get_few_shot_examples(train_df)

# ==========================================
# 3. Predict function
# ==========================================
def predict_label(tweet_text, mode="zero-shot"):
    """
    mode: "zero-shot", "zero-shot-with-context", "few-shot", "few-shot-with-context"
    """
    if mode == "few-shot-with-context":
        prompt = f"{context_info}\n{few_shot_examples}\nNow classify this tweet:\nTweet: {tweet_text}\nLabel:"
    elif mode == "zero-shot-with-context":
        prompt = f"{context_info}\nNow classify this tweet:\nTweet: {tweet_text}\nLabel:"
    elif mode == "few-shot":
        prompt = f"{few_shot_examples}\nNow classify this tweet:\nTweet: {tweet_text}\nLabel:"
    else:
        # Zero-shot
        prompt = f"Classify this tweet:\nTweet: {tweet_text}\nLabel:"
        
    try:
        response = model_norm.generate_content(prompt)
        res_json = json.loads(response.text)
        return res_json.get("label", "Unclassified")
    except Exception as e:
        return "Error"

# ==========================================
# 4. Prediction
# ==========================================
modes = ["zero-shot", "zero-shot-with-context", "few-shot", "few-shot-with-context"]
evaluation_results = {}

for mode in modes:
    print(f"\n>>> Running Evaluation: {mode.upper()} Mode...")
    y_pred = []
    y_true = test_df['label'].tolist()
    
    for i, tweet in enumerate(test_df['tweet']):
        pred = predict_label(tweet, mode=mode)
        y_pred.append(pred)
        
        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{len(test_df)} samples...")
        
        time.sleep(1)
        
    evaluation_results[mode] = y_pred

# ==========================================
# 5. Evaluation: Accuracy, Precision, Recall, F1
# ==========================================
for mode in modes:
    print("\n" + "="*50)
    print(f"REPORT FOR: {mode.upper()}")
    print("="*50)
    
    y_true = test_df['label'].tolist()
    y_pred = evaluation_results[mode]
    
    # Precision, Recall, F1-score
    report = classification_report(y_true, y_pred, zero_division=0)

    print("\nDetailed Metrics (Precision, Recall, F1):")
    print(report)